In [17]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import tensorflow.keras as keras
import ipywidgets as widgets
from IPython.display import display

from sklearn.model_selection import train_test_split

In [15]:
data = np.load("../data/dsprites.npz", allow_pickle=True, encoding="latin1")

imgs = data["imgs"]
latents_values = data["latents_values"]
latents_classes = data["latents_classes"]

X_train, X_test = train_test_split(imgs, test_size=0.6, random_state=2) # temporarily take small subset of images
print(f"X train shape: {X_train.shape}")

X train shape: (294912, 64, 64)


In [ ]:
np.random.seed(2)
indices = np.random.choice(len(imgs), size=10, replace=False)

def show_sprite(i):
    idx = indices[i]
    plt.figure(figsize=(3,3))
    plt.imshow(imgs[idx], cmap="gray")
    plt.title(f"Dataset index: {idx}")
    plt.axis("off")
    plt.show()

slider = widgets.IntSlider(value=0, min=0, max=9, step=1, description="Sample")

widgets.interact(show_sprite, i=slider)

interactive(children=(IntSlider(value=0, description='Sample', max=9), Output()), _dom_classes=('widget-intera…

<function __main__.show_sprite(i)>

In [ ]:
# variational Autoencoder architecture

#custom sampling layers
class Sampling(keras.layers.Layer):
    def call(self, inputs):
        mu, log_var = inputs
        eps = tf.random.normal(shape=tf.shape(mu))
        return mu + tf.exp(0.5 * log_var) * eps
    
class Decoder(keras.Model):
    def __init__(self, original_dim):
        super().__init__()
        # define encoder layers
        self.dropout = keras.layers.Dropout(0.2)
        self.layer1 = keras.layers.Dense(128, activation="relu")
        self.layer2 = keras.layers.Dense(128, activation="relu")
        self.layer3 = keras.layers.Dense(256, activation="relu")
        self.output_layer = keras.layers.Dense(original_dim)

    def call(self, z):
        x = self.layer1(z)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.dropout(x)
        x = self.layer3(x)
        x = self.output_layer(x)
        return x
    
class Encoder(keras.Model):
    def __init__(self, latent_dim=6):
        super().__init__()
        self.flatten = keras.layers.Flatten()
        self.dropout = keras.layers.Dropout(0.2)
        self.layer1 = keras.layers.Dense(256, activation="relu")
        self.layer2 = keras.layers.Dense(128, activation="relu")
        self.layer3 = keras.layers.Dense(128, activation="relu")
        self.mu_layer = keras.layers.Dense(latent_dim)
        self.log_var_layer = keras.layers.Dense(latent_dim)
        self.sampling = Sampling()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.layer1(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.dropout(x)
        x = self.layer3(x)
        mu = self.mu_layer(x)
        log_var = self.log_var_layer(x)
        z = self.sampling((mu, log_var))
        return mu, log_var, z

class VariationalAutoencoder(keras.Model):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        #TODO: rest of init code
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(input_dim)

        # track the losses
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.recon_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")
    
    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.recon_loss_tracker,
            self.kl_loss_tracker,
        ]

    def call(self, x, training=False):
        mu, log_var, z = self.encoder(x, training=training)
        reconstruction = self.decoder(z, training=training)
        return reconstruction
    
    def train_step(self, data):
        if isinstance(data, tuple):
            x = data[0]
        else:
            x = data

        with tf.GradientTape() as tape:
            mu, log_var, z = self.encoder(x, training=True)
            reconstruction = self.decoder(z, training=True)

            x_flat = tf.reshape(x, [tf.shape(x)[0], -1])

            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.binary_crossentropy(x_flat, reconstruction)
                )
            )

            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + log_var - tf.square(mu) - tf.exp(log_var),
                    axis=1
                )
            )

            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.recon_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

In [51]:
# first lets try it on MNIST

(mnist_train, _), (mnist_test, _) = keras.datasets.mnist.load_data()

X_dim = mnist_train.shape[-1] * mnist_train.shape[-2]

#normalise the images
mnist_train = mnist_train.astype("float32") / 255.0
mnist_test = mnist_test.astype("float32") / 255.0

latent_dim = 10

model = VariationalAutoencoder(latent_dim=latent_dim, input_dim=X_dim)
model.compile(optimizer="adam")
model.fit(mnist_train, epochs = 50, batch_size=32)

Epoch 1/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - kl_loss: 0.0257 - loss: 13.5153 - reconstruction_loss: 13.4896
Epoch 2/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - kl_loss: 0.0031 - loss: 13.0752 - reconstruction_loss: 13.0721
Epoch 3/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - kl_loss: 9.7793e-04 - loss: 12.9953 - reconstruction_loss: 12.9943
Epoch 4/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - kl_loss: 5.8971e-04 - loss: 12.8273 - reconstruction_loss: 12.8267
Epoch 5/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - kl_loss: 2.9608e-06 - loss: 12.6553 - reconstruction_loss: 12.6553
Epoch 6/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - kl_loss: 4.9169e-06 - loss: 12.6324 - reconstruction_loss: 12.6324
Epoch 7/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - kl_loss: 1.1786e-06 - loss: 12.4158 - reconstruction_loss: 12.4158
Epoch 8/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - kl_loss: 4.7491e-06 - loss: 12.4153 - reconstruction_loss: 12.4153
Epoch 9/50
1875/1875 ━━━

In [57]:
# generate new samples
N_new = 20
# sample new std normal z ~ N(shape:latent_dim)
z = tf.random.normal(shape=(N_new, 10))
X_recon = model.decoder(z)
# feed through models trained decoder
X_recon = X_recon.numpy().reshape(N_new, 28, 28)

In [58]:
def show_generated(i):
    plt.figure(figsize=(4,4))
    plt.imshow(X_recon[i], cmap="gray")
    plt.axis("off")
    plt.title(f"Generated sample {i}")
    plt.show()

widgets.interact(show_generated, i=(0, N_new-1));

interactive(children=(IntSlider(value=9, description='i', max=19), Output()), _dom_classes=('widget-interact',…

In [ ]:
X_recon_n.reshape()